In [1]:
#This program reads dataset tables from NCBI Genome searches and sorts extremophilic candidates and control candidates
##This program will also analyze sequence length and contig N50 values
###This program will compile the selection of extremophilic and control candidates

In [3]:
import pandas as pd

#loading the NCBI genome search result table
BacillaceaeTable = pd.read_csv("/content/drive/MyDrive/Capstone/Data/BacillaceaeTable.tsv", sep='\t')
print(BacillaceaeTable.columns)
print(BacillaceaeTable.head())
print("\n", "There are ", len(BacillaceaeTable), "species genomes in the NCBI search")

Index(['Organism Scientific Name', 'Organism Common Name',
       'Organism Qualifier', 'Taxonomy id', 'Assembly Name',
       'Assembly Accession', 'Source', 'Annotation', 'Level', 'Contig N50',
       'Size', 'Submission Date', 'Gene Count', 'BioProject', 'BioSample'],
      dtype='object')
            Organism Scientific Name Organism Common Name  Organism Qualifier  \
0  Alkalihalophilus pseudofirmus OF4                  NaN         strain: OF4   
1         Bacillus cereus ATCC 14579                  NaN  strain: ATCC 14579   
2       Bacillus anthracis str. Ames                  NaN        strain: Ames   
3         Bacillus cereus ATCC 10987                  NaN  strain: ATCC 10987   
4     Bacillus anthracis str. Sterne                  NaN      strain: Sterne   

   Taxonomy id Assembly Name Assembly Accession  Source  \
0       398511      ASM582v2    GCF_000005825.2  RefSeq   
1       226900      ASM782v1    GCF_000007825.1  RefSeq   
2       198094      ASM784v1    GCF_000007

In [4]:
#Next we will compare the NCBI Genome search results with the pre-compiled list of Bacillaceae thermophiles

#Read in the Bacillaceae list
dfBacillaceae = pd.read_csv("/content/drive/MyDrive/Capstone/Data/listBacillaceae.txt", header=None, sep="|")
listBacillaceae = dfBacillaceae[0].dropna().str.strip().str.lower().tolist()
print(len(listBacillaceae), " thermophile species have been imported")
print(listBacillaceae[0:10])


93  thermophile species have been imported
['calditerricola satsumensis', 'calditerricola yamamurae', 'geobacillus sp wch70', 'geobacillus thermocatenulatus', 'geobacillus kaustophilus hta426', 'geobacillus stearothermophilus', 'geobacillus galactosidasius', 'geobacillus sp y4.1mc1', 'geobacillus sp y412mc52', 'parageobacillus caldoxylosilyticus']


In [5]:
#Flexible matching between NCBI Bacillaceae Genome search and the Bacillaceae thermophiles to group extremophiles and control species

#Creating the matching & assignment function
def match_assignment(ncbiName):
  ncbiFullName = str(ncbiName.lower().strip()) #basic, cleaned ncbi name
  ncbiSplitName = ncbiFullName.split() #this is splitting the ncbi sci name into single words (genus and species)
  ncbiSimpleName = " ".join(ncbiSplitName[:2]) if len(ncbiSplitName) >= 2 else ncbiFullName #brings together the genus and species while ommitting strains

  for species in listBacillaceae:
    listFullName = str(species).lower().strip() #basic cleaned list name
    listSplitName = listFullName.split() #this is splitting the list sci name into single words (genus and species)
    listSimpleName = " ".join(listSplitName[:2]) if len(listSplitName) >= 2 else listFullName
    if (listFullName in ncbiFullName) or (ncbiFullName in listFullName) or (listSimpleName == ncbiSimpleName):
      return "Extremophile"

  return "Mesophile"

#Execute matching function
BacillaceaeTable["Assignment"] = BacillaceaeTable["Organism Scientific Name"].apply(match_assignment)

#Display organism assignments
print("Below are the number of species assigned to each group", "\n")
print(BacillaceaeTable["Assignment"].value_counts())


Below are the number of species assigned to each group 

Assignment
Mesophile       2985
Extremophile     112
Name: count, dtype: int64


In [6]:
#Isolate all unique species with the Extremophile assignment/create the thermophile genome group

BacillaceaeExtremos = BacillaceaeTable[BacillaceaeTable["Assignment"] == "Extremophile"].copy()
BacillaceaeExtremos = BacillaceaeExtremos.drop_duplicates(subset=["Organism Scientific Name"])
BacillaceaeExtremos[["Assignment", "Organism Scientific Name", "Size", "Contig N50", "Gene Count", "Assembly Accession"]]


,Assignment,Organism Scientific Name,Size,Contig N50,Gene Count,Assembly Accession
5,Extremophile,Bacillus licheniformis DSM 13 = ATCC 14580,4222645,4222645,4376.0,GCF_000008425.1
107,Extremophile,Bacillus methanolicus MGA3,3425208,3337035,3480.0,GCF_000724485.1
229,Extremophile,Bacillus licheniformis,4252398,4252398,4431.0,GCF_001596055.1
533,Extremophile,Bacillus halotolerans,4154245,4154245,4286.0,GCF_004006435.1
559,Extremophile,Neobacillus thermocopriae,2843874,2843874,3011.0,GCF_004116995.1
1058,Extremophile,Caldalkalibacillus thermarum TA2.A1,3344914,3344914,3486.0,GCF_019856495.1
1277,Extremophile,Bacillus methanolicus,3560650,3540451,3681.0,GCF_023278615.1
1782,Extremophile,Aeribacillus composti,3834459,3834459,3727.0,GCF_031760945.1
2099,Extremophile,Caldifermentibacillus hisashii,3885383,3885383,3790.0,GCF_039654125.1
2180,Extremophile,Metabacillus halosaccharovorans,5833127,5378041,5734.0,GCF_040959945.1


In [7]:
#Removing the strain variants from the extremophile group
BacillaceaeExtremosUnique = BacillaceaeExtremos.drop([107, 5])
BacillaceaeExtremosUnique[["Assignment", "Organism Scientific Name", "Size", "Contig N50", "Gene Count", "Assembly Accession"]]

,Assignment,Organism Scientific Name,Size,Contig N50,Gene Count,Assembly Accession
229,Extremophile,Bacillus licheniformis,4252398,4252398,4431.0,GCF_001596055.1
533,Extremophile,Bacillus halotolerans,4154245,4154245,4286.0,GCF_004006435.1
559,Extremophile,Neobacillus thermocopriae,2843874,2843874,3011.0,GCF_004116995.1
1058,Extremophile,Caldalkalibacillus thermarum TA2.A1,3344914,3344914,3486.0,GCF_019856495.1
1277,Extremophile,Bacillus methanolicus,3560650,3540451,3681.0,GCF_023278615.1
1782,Extremophile,Aeribacillus composti,3834459,3834459,3727.0,GCF_031760945.1
2099,Extremophile,Caldifermentibacillus hisashii,3885383,3885383,3790.0,GCF_039654125.1
2180,Extremophile,Metabacillus halosaccharovorans,5833127,5378041,5734.0,GCF_040959945.1
2803,Extremophile,Salinibacillus aidingensis,3522017,3522017,3599.0,GCF_055393355.1


In [8]:
#Filtering out the extremophiles to create the mesophile/control group to compare
BacillaceaeMesos = BacillaceaeTable[BacillaceaeTable["Assignment"] == "Mesophile"].copy()

#Selecting mesophiles that are in the same genuses as the extremophiles
extremeGenera = BacillaceaeExtremosUnique["Organism Scientific Name"].str.split().str[0].str.lower()
BacillaceaeMesos["Genus"] = BacillaceaeMesos["Organism Scientific Name"].str.split().str[0].str.lower()
matchedMesos = BacillaceaeMesos[BacillaceaeMesos["Genus"].isin(extremeGenera)]

print(extremeGenera, "\n")
print("There are ", len(matchedMesos), " mesophiles that match the extremophile genera")


229                  bacillus
533                  bacillus
559               neobacillus
1058       caldalkalibacillus
1277                 bacillus
1782             aeribacillus
2099    caldifermentibacillus
2180             metabacillus
2803           salinibacillus
Name: Organism Scientific Name, dtype: object 

There are  2433  mesophiles that match the extremophile genera


In [9]:
#selecting the control group from the matchedMesos df randomly
x=0

#This loop iterates over the selected thermophiles and chooses mesophiles in the same genus, if available.
while x<=23:
  mesophileControls = []
  for genus in extremeGenera:
    genusPool = matchedMesos[matchedMesos["Genus"] == genus]
    #Condition 1: greater than 3 matches in the same genus
    if len(genusPool) >= 3:
        randomSample = genusPool.sample(n=3)
    #Condition 2: some matches, but fewer than 3 (take them all)
    elif len(genusPool) > 0:
        randomSample = genusPool
    #Condition 3: No mesophiles in that genus. Random selection of three
    else:
        randomSample = BacillaceaeMesos.sample(n=3)
    mesophileControls.append(randomSample)

  #Put all the mesophile controls in one list, remove duplicates
  mesophileControls = pd.concat(mesophileControls)
  mesophileControls = mesophileControls.drop_duplicates(subset=["Organism Scientific Name"])

  x = len(mesophileControls)
print("There are ", x, " mesophiles in the control group")

There are  24  mesophiles in the control group


In [10]:
mesophileControls

,Organism Scientific Name,Organism Common Name,Organism Qualifier,Taxonomy id,Assembly Name,Assembly Accession,Source,Annotation,Level,Contig N50,Size,Submission Date,Gene Count,BioProject,BioSample,Assignment,Genus
1241,Bacillus thuringiensis serovar morrisoni str. ...,NaN,strain: 4AA1,1461024,ASM2281072v1,GCF_022810725.1,RefSeq,GCF_022810725.1-RS_2025_06_08,Complete Genome,5669783,6313491,2022-04-04,6517.0,PRJNA227493,SAMN03890260,Mesophile,bacillus
2204,Bacillus velezensis,NaN,"isolate: 1, strain: CM35",492670,064049E_CM35,GCF_041695445.1,RefSeq,GCF_041695445.1-RS_2024_08_31,Complete Genome,4240818,4240818,2024-08-28,4256.0,PRJNA1142391,SAMN42934666,Mesophile,bacillus
30,Bacillus thuringiensis BMB171,NaN,strain: BMB171,714359,ASM9216v1,GCF_000092165.1,RefSeq,GCF_000092165.1-RS_2026_05_17,Complete Genome,5330088,5643051,2010-05-20,5770.0,PRJNA43631,SAMN02603314,Mesophile,bacillus
23,Bacillus cereus G9842,NaN,strain: G9842,405531,ASM2130v1,GCF_000021305.1,RefSeq,GCF_000021305.1-RS_2025_08_25,Complete Genome,5387334,5736823,2008-12-19,5897.0,PRJNA17733,SAMN02604060,Mesophile,bacillus
1921,Bacillus sp. PS210,NaN,isolate: PS210,2954721,PS02022,GCF_037963545.1,RefSeq,GCF_037963545.1-RS_2024_04_05,Complete Genome,4103372,4103372,2024-04-04,4253.0,PRJNA768416,SAMN29228873,Mesophile,bacillus
1779,Bacillus tropicus,NaN,strain: T36S-23,2026188,ASM3135704v1,GCF_031357045.1,RefSeq,GCF_031357045.1-RS_2025_09_12,Complete Genome,5262398,5319187,2023-09-12,5534.0,PRJNA944409,SAMN33748034,Mesophile,bacillus
3001,Neobacillus sp. DHD-1R,NaN,strain: DHD-1R,3711415,ASM5774799v1,GCF_057747995.1,RefSeq,GCF_057747995.1-RS_2026_05_26,Complete Genome,5411504,5411504,2026-05-25,5342.0,PRJNA1466056,SAMN59905824,Mesophile,neobacillus
530,Neobacillus mesonae,NaN,strain: H20-5,1193713,ASM399973v1,GCF_003999735.1,RefSeq,GCF_003999735.1-RS_2026_04_27,Complete Genome,5839313,5839313,2019-01-08,5698.0,PRJNA395788,SAMN07414821,Mesophile,neobacillus
2324,Neobacillus oceani,NaN,strain: SCS-31,3115292,ASM4623759v1,GCF_046237595.1,RefSeq,GCF_046237595.1-RS_2025_02_16,Complete Genome,4577160,4610304,2024-12-20,4632.0,PRJNA1063218,SAMN39489481,Mesophile,neobacillus
2809,Caldalkalibacillus mannanilyticus,NaN,strain: JCM 10596,1418,ASM5539756v1,GCF_055397565.1,RefSeq,GCF_055397565.1-RS_2026_02_24,Complete Genome,4618642,4618642,2025-08-27,4408.0,NaN,SAMD01611095,Mesophile,caldalkalibacillus


In [11]:
#Finally exporting all the lists of species

#The separate data tables for each set of genomes
BacillaceaeExtremosUnique.to_csv("/content/drive/MyDrive/Capstone/Data/bacillaceaeExtremophiles.csv", index=False)
mesophileControls.to_csv("/content/drive/MyDrive/Capstone/Data/bacillaceaeMesophiles.csv", index=False)

#List of all assembly accession numbers in both sets
allAccessions = pd.concat([BacillaceaeExtremosUnique[["Assembly Accession"]],
    mesophileControls[["Assembly Accession"]]])
allAccessions.to_csv("/content/drive/MyDrive/Capstone/Data/allBacillaceaeAccessions.txt", index=False, header=False, sep='\n')